<a href="https://colab.research.google.com/github/Song-yiJung/korean-ocr-lectures/blob/main/2026-08-aks-lecture/01_step1_vision/step1_vision_colab_20260813.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Step 1 — Vision OCR로 1차 텍스트 뽑기**

*2026 한국학중앙연구원 OCR 강의 실습*

사료 이미지에서 글자를 뽑아내는 단계이다. 이미지를 Google Cloud Vision에 한 장씩 보내고, 돌아온 결과를 파일로 저장한다.

**직접 고치는 셀은 ③ 하나뿐이다.** 나머지 제목에는 `[실행만]`이 붙어 있다. 읽지 않아도 되고 눌러서 실행만 하면 된다.

| 셀 | 하는 일 | 손대는가 | 시간 |
|---|---|---|---|
| ① | 프로그램 설치 | 실행만 | 30초 |
| ② | Drive 연결 | 실행만 | 20초 |
| **③** | **설정** | **★ 고친다** | — |
| ③-보충 | 사료 3장 받기 | 실행만 | 10초 |
| ④ | 인증 | 실행만 | 5초 · **확인 1** |
| ⑤ | 함수 만들기 | 실행만 | 즉시 |
| ⑥ | 대상 모으기 | 실행만 | 즉시 · **확인 2** |
| ⑦ | OCR 실행 | 실행만 | 약 10초 |
| ⑧ | 결과 보기 | 실행만 | 즉시 · **확인 3** |
| ⑨ | *[참고]* 신뢰도 살펴보기 | 실행만 | 즉시 · 건너뛸 수 있음 |

**확인**이 세 곳 있다. 그 셀의 출력이 설명과 같은지 보고 넘어간다. 다르면 그 자리에서 멈춘다. ⑨는 시간이 남을 때만 한다.

## 시작 전 — 사본 저장

지금 노트북은 **읽기 전용**이다. 상단 **`파일 → Drive에 사본 저장`**을 누른다. 새 탭이 열리고, **이후 작업은 모두 그 사본에서** 한다.

준비물은 **Vision API 키 파일(`.json`)** 하나다. 발급 절차는 [API 키 발급 안내](https://github.com/Song-yiJung/korean-ocr-lectures/blob/main/docs/api-key-setup.md)에 있다. 사료 이미지는 준비하지 않아도 된다. 3장을 ③-보충에서 받는다.

3장 기준 약 5분. Vision은 월 1,000회까지 무료라 실습에서는 비용이 들지 않는다.

## 코드를 읽는 법

코드를 쓸 줄 몰라도 실습은 된다. 다만 화면에 무엇이 지나가는지 알면, 오류가 났을 때 어디를 볼지 짚을 수 있다.

**먼저 여섯 가지다.** 이 여섯만 알면 셀이 무슨 일을 하는지 읽힌다.

| 보이는 것 | 뜻 |
|---|---|
| `!pip install ...` | `!`로 시작하면 파이썬이 아니라 **설치 명령**이다. 처음 한 번만 실행한다. |
| `import os` | **라이브러리**, 곧 남이 만들어 둔 도구 묶음을 불러온다. 이 줄이 없으면 그 도구를 못 쓴다. |
| `SLEEP_SEC = 6` | **값에 이름을 붙인다.** 아래 셀에서 그 이름으로 다시 꺼내 쓴다. |
| `# 뒤의 글` | **주석**이다. 사람이 읽으라고 적어 둔 것이고 실행되지 않는다. |
| `read_image(path)` | 괄호가 붙으면 **부르는 것**이다. 이것을 함수 호출이라 한다. |
| 칸 왼쪽의 **▶** | 누르면 **그 칸만** 실행된다. 위에서 아래로 차례대로 누른다. |

**여기에 넷이 더 나온다.** 뜻만 알아 두면 된다.

| 보이는 것 | 뜻 |
|---|---|
| `def read_image(path):` | **절차에 이름을 붙인다.** 만들어만 두고 실행은 하지 않는다. |
| `for name in images:` | 목록에서 하나씩 꺼내 **같은 일을 반복한다.** 사료가 여러 장이라 필요하다. |
| `if ... :` | 조건이 맞을 때만 그 아래를 한다. |
| `print(...)` | 화면에 찍는다. 결과에는 영향을 주지 않는다. |

**들여쓰기가 곧 범위다.** `for` 아래로 들여쓴 줄들이 반복되는 몸통이다. 파이썬은 괄호가 아니라 여백으로 묶음을 표시한다.

담는 그릇은 셋이 나온다. `'글자'`는 **문자열**, `[...]`는 여러 개를 순서대로 담는 **목록**, `{'이름': 값}`은 이름표를 붙여 담는 **사전**이다. 사전을 파일로 저장한 것이 곧 **JSON**이고, 이 노트북의 결과물이 그것이다.


### ① [실행만] 프로그램 설치

Vision을 부르는 데 필요한 프로그램을 설치한다.

`google-cloud-vision`은 **구글이 만들어 배포하는 파이썬 묶음**이다. 우리가 이미지를 인터넷 너머 구글 서버로 보내고 결과를 받아 오는 복잡한 절차를, 함수 몇 개로 줄여 준다. 코랩에는 기본으로 깔려 있지 않아 매번 설치한다.

`-q`는 설치 과정을 조용히 하라는 뜻이다. 30초쯤 걸리고, 오류 없이 멈추면 정상이다.

In [ ]:
!pip install -q google-cloud-vision

### ② [실행만] Google Drive 연결

코랩은 **잠깐 빌려 쓰는 컴퓨터**라, 창을 닫으면 안에 만든 파일이 사라진다. 그래서 본인 Drive를 연결해 두고 사료와 결과를 거기에 둔다.

실행하면 권한 요청 창이 뜬다. 계정을 고르고 **허용**을 누른다.

```
Mounted at /content/drive
```

이후 Drive의 `내 드라이브`는 `/content/drive/MyDrive/` 라는 경로로 보인다.

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

### ③ ★ 여기만 고친다 — 설정

이 노트북에서 손대는 곳은 아래 하나뿐이다. **강의에서는 [1]만 확인하고, 나머지는 그대로 두고 실행한다.**

아래 코드는 전부 `이름 = 값` 꼴이다. 값에 이름을 붙여 두면 뒤 셀들이 그 이름으로 꺼내 쓴다. 그래서 경로를 바꾸려면 여기 한 곳만 고치면 된다.

---

#### [1] `VISION_KEY_SECRET_NAME` — 키 파일 위치를 적어 둘 이름

키 파일 경로를 코드에 그대로 쓰면 노트북을 공유할 때 노출된다. 코랩에는 이런 값을 코드와 따로 보관하는 **보안 비밀**이 있다. 여기 넣는 것은 키 파일의 *내용*이 아니라 *파일이 있는 경로*다.

**이 작업은 코드가 아니라 화면 왼쪽에서 한다.**

1. 받은 `.json` 키 파일을 Drive에 올린다. 예: `내 드라이브/keys/`
2. 그 파일의 경로를 확인한다. 보통 `/content/drive/MyDrive/keys/vision_key.json` 꼴이다.
3. 화면 **맨 왼쪽 세로 막대의 🔑 아이콘**을 누른다.
4. **＋ 새 보안 비밀**을 누른다.
5. **이름**에 `VISION_KEY_PATH` 라고 적는다. *(아래 코드의 값과 글자 하나까지 같아야 한다.)*
6. **값**에 2번 경로를 붙여 넣는다.
7. **노트북 액세스**를 **켠다.** 켜지 않으면 ④에서 반드시 막힌다.

> step2의 Gemini 키는 파일이 아니라 `AIzaSy…`로 시작하는 **문자열**이다. 그때는 경로가 아니라 키 문자열 자체를 값에 넣는다. 두 키의 생김새가 다르다는 점이 가장 흔한 혼동거리다.

---

#### [2]~[4] 나머지 — 강의에서는 그대로 둔다

`INPUT_DIR`은 사료를 둘 폴더, `OUTPUT_DIR`은 결과를 저장할 폴더다. `LANGUAGE_HINTS`는 사료의 주요 언어로, 실습 사료가 일본어·구자체 한자라 `['ja', 'zh-Hant']`로 두었다. 이것은 Vision에게 주는 **힌트**이지 명령이 아니다. 2~3개가 적당하고, 많이 넣으면 오히려 정확도가 떨어진다.

강의가 끝난 뒤 본인 사료에 쓸 때만 고친다. 한글은 `['ko', 'en']`, 한문은 `['zh-Hant']`.

In [ ]:
# ③ 설정 — 강의에서는 [1]만 확인하고 실행한다

# [1] 왼쪽 🔑 에 등록한 보안 비밀의 이름
VISION_KEY_SECRET_NAME = 'VISION_KEY_PATH'

# [2] 사료 이미지를 둘 폴더
INPUT_DIR = '/content/drive/MyDrive/사료_이미지_강의'

# [3] 결과를 저장할 폴더
OUTPUT_DIR = '/content/drive/MyDrive/vision_결과_강의'

# [4] 사료의 주요 언어
LANGUAGE_HINTS = ['ja', 'zh-Hant']

print('이미지 폴더:', INPUT_DIR)
print('결과 폴더  :', OUTPUT_DIR)

### ③-보충 [실행만] 실습 사료 3장 받기

실습 사료 3장을 위 `INPUT_DIR` 폴더로 가져온다. **③을 먼저 실행한 뒤에 누른다.**

「기부 물건에 관한 건」(昭和 10년) 한 문건의 세 면이다. 같은 문건 안에서도 결과가 얼마나 달라지는지 보려고 고른 것이다.

| 파일 | 면의 성격 | 볼 것 |
|---|---|---|
| `E006-005-001-001` | 기안문 표지 — 붉은 서식 + 필사 | 오독과 환각이 함께 나타난다 |
| `E006-005-001-002` | 이면 — 여백 위주, 흐린 필사 | 거의 잡히지 않는 면을 어떻게 볼 것인가 |
| `E006-005-002-001` | 별지 통첩 — 인쇄 활자 | 조건이 좋을 때. 앞 두 장과 대조된다 |

조선총독부박물관 문서, 국립중앙박물관 소장. 교육 목적으로 쓴다.

```
복사: E006-005-001-001.jpg
복사: E006-005-001-002.jpg
복사: E006-005-002-001.jpg
3 장을 /content/drive/MyDrive/사료_이미지_강의 에 두었습니다.
```

코드에서 볼 것은 `for` 한 곳이다. **목록에 담긴 파일 이름을 하나씩 꺼내 같은 복사 작업을 세 번 반복한다.** 이 구조가 ⑦에서 그대로 다시 나온다. 거기서는 복사 대신 OCR을 반복할 뿐이다.

> 본인 사료로 작업할 때는 **이 셀을 건너뛰고** 이미지를 `INPUT_DIR`에 직접 올린다.

In [ ]:
# 강의 저장소에서 실습 사료를 내려받는다
!rm -rf /content/_repo
!git clone --depth 1 -q https://github.com/Song-yiJung/korean-ocr-lectures.git /content/_repo

import os
import shutil

os.makedirs(INPUT_DIR, exist_ok=True)

samples_dir = '/content/_repo/2026-08-aks-lecture/samples'
names = ['E006-005-001-001.jpg', 'E006-005-001-002.jpg', 'E006-005-002-001.jpg']

for name in names:
    shutil.copy(samples_dir + '/' + name, INPUT_DIR)
    print('복사:', name)

print(len(names), '장을', INPUT_DIR, '에 두었습니다.')

### ④ [실행만] 인증 — **확인 1**

③에서 등록한 보안 비밀을 읽어 Vision에 연결한다.

**아래 두 줄이 나오면 통과다.**

```
키 파일: /content/drive/MyDrive/keys/vision_key.json
Vision 연결 완료
```

**여기가 가장 많이 막히는 자리다.** 원인은 대개 셋 중 하나다.

* 보안 비밀 이름이 `VISION_KEY_PATH`와 다르다 — 대소문자·밑줄까지 같아야 한다
* **노트북 액세스**가 꺼져 있다
* 값에 넣은 경로에 실제로 파일이 없다 — `키 파일을 찾지 못했습니다`가 나온다

고쳤으면 ④를 다시 누른다. `SecretNotFoundError`라는 붉은 오류가 나면 이름이 다르거나 액세스가 꺼진 것이다.

코드는 네 가지를 한다. 보안 비밀에서 경로를 **꺼내고**, 그 경로를 Vision이 찾아볼 자리에 **넣어 두고**, 파일이 실제 있는지 `if`로 **확인하고**, 연결 창구인 `client`를 **만든다.** 이 `client`가 이후 구글과 이야기하는 통로가 되며 ⑤에서 다시 쓴다.

In [ ]:
import os
from google.colab import userdata
from google.cloud import vision

# 보안 비밀에서 키 파일 경로를 꺼낸다
key_path = userdata.get(VISION_KEY_SECRET_NAME)
print('키 파일:', key_path)

# Vision이 찾아볼 자리에 그 경로를 넣어 둔다
os.environ['GOOGLE_APPLICATION_CREDENTIALS'] = key_path

if os.path.exists(key_path):
    client = vision.ImageAnnotatorClient()
    print('Vision 연결 완료')
else:
    print('키 파일을 찾지 못했습니다. 보안 비밀의 값(경로)을 확인하세요.')

### ⑤ [실행만] 함수 만들기

**이 셀은 아무 일도 하지 않는다.** 이미지 한 장을 처리하는 절차에 `read_image`라는 이름을 붙여 둘 뿐이다. 실제 실행은 ⑦에서 이 이름을 부를 때 일어난다.

```
read_image 준비 완료
```

절차에 이름을 붙여 두는 이유는 **사료가 여러 장이기 때문**이다. 3장이면 세 번, 900장이면 900번 같은 일을 한다. 그때마다 같은 코드를 다시 쓰는 대신, 한 번 만들어 두고 이름만 부른다.

`def read_image(path):`에서 `path`는 **받을 자리**다. ⑦에서 `read_image('.../E006-005-001-001.jpg')`라고 부르면 그 경로가 `path`에 들어간다.

부르는 `client.document_text_detection`이 이 노트북의 핵심이다. Vision에는 글자를 읽는 방식이 둘 있다.

* `text_detection` — 간판·표지판처럼 글자가 흩어져 있는 사진용
* `document_text_detection` — **문서용.** 행과 단락이 있는 것을 전제로 읽는다

사료는 글자가 빽빽하고 행이 분명하므로 뒤엣것을 쓴다. 같은 이미지라도 어느 쪽을 부르느냐에 따라 결과가 달라진다.

---

**맨 끝 `return`이 두 개를 돌려준다.**

```python
return text, raw
```

`text`는 우리가 읽을 본문이고, `raw`는 **구글이 보내 준 답장 전체**다. 답장에는 본문 말고도 글자마다 **이미지의 어느 자리에 있었는지**와 **모델이 얼마나 확신하는지**가 들어 있다. 지금은 쓰지 않지만 버리지 않고 저장해 둔다. 한 번 뽑은 결과를 다시 얻으려면 비용이 들기 때문이다. 무엇이 들었는지는 ⑨에서 본다.

In [ ]:
def read_image(path):
    # 이미지 파일을 열어 통째로 읽는다
    content = open(path, 'rb').read()

    image = vision.Image(content=content)
    context = vision.ImageContext(language_hints=LANGUAGE_HINTS)

    # 구글에 보내고 답장을 받는다 (문서용 방식)
    response = client.document_text_detection(image=image, image_context=context)

    if response.error.message:
        raise Exception(response.error.message)

    text = response.full_text_annotation.text.strip()   # 본문만
    raw = vision.AnnotateImageResponse.to_json(response)  # 답장 전체

    return text, raw


print('read_image 준비 완료')

### ⑥ [실행만] 대상 모으기 — **확인 2**

`INPUT_DIR` 폴더를 살펴 이미지 파일만 골라 목록으로 만든다. 실제 호출 직전에 몇 장을 처리할지 확인하는 자리다.

**반드시 `3 장`이어야 한다.**

```
처리 대상: 3 장
  E006-005-001-001.jpg
  E006-005-001-002.jpg
  E006-005-002-001.jpg
```

`0 장`이면 ③-보충을 실행하지 않았거나 `INPUT_DIR`이 틀렸다. **3장보다 많으면** 폴더에 다른 이미지가 섞인 것이니 다음 셀을 누르기 전에 알린다.

코드는 `for`와 `if`가 맞물린 꼴이다. 폴더 안 이름을 하나씩 꺼내(`for`), 이미지 확장자로 끝나는 것만(`if`) 빈 목록에 담는다(`append`). 그래서 폴더에 메모나 엑셀 파일이 섞여 있어도 걸러진다.

In [ ]:
import os

images = []

for name in sorted(os.listdir(INPUT_DIR)):
    if name.lower().endswith(('.jpg', '.jpeg', '.png', '.tif', '.tiff')):
        images.append(name)

print('처리 대상:', len(images), '장')

for name in images:
    print(' ', name)

### ⑦ [실행만] OCR 실행

**여기가 실제로 글자를 뽑는 셀이다.** 3장이면 10초쯤 걸리고, 한 장씩 진행이 찍힌다.

```
[1/3] E006-005-001-001.jpg
   -> 412 자
[2/3] E006-005-001-002.jpg
   -> 23 자
[3/3] E006-005-002-001.jpg
   -> 1038 자

완료: 3 장 처리
```

글자 수는 조금씩 다르게 나온다. **두 번째 장이 유난히 적은 것은 고장이 아니다.** 흐린 필사면이라 원래 잘 잡히지 않는다. 세 장의 차이를 보는 것이 이 실습의 목적이다.

**다시 눌러도 안전하다.** `if os.path.exists(json_path)` 줄이 이미 결과가 있는 장을 건너뛴다. 중간에 끊겨도 다시 실행하면 남은 것만 이어서 한다.

---

**장마다 파일이 둘 생긴다.**

| 파일 | 안에 든 것 | 언제 여나 |
|---|---|---|
| `E006-005-001-001.json` | 뽑은 텍스트 | **⑧에서 연다** |
| `E006-005-001-001_full.json` | 구글 답장 전체 (좌표·신뢰도) | 열지 않는다. ⑨에서 들여다본다 |

둘째 파일은 수백 KB짜리 숫자 덩어리라 사람이 읽을 것이 못 된다. **그래서 따로 뺐다.** 첫째 파일은 그만큼 짧아서 열어 볼 수 있다.

첫째 파일 안은 이렇게 생겼다.

```
{ "file_path": ..., "vision_raw": ..., "gemini_corrected": "" }
```

`vision_raw`가 방금 뽑은 1차 텍스트고, `gemini_corrected`는 **일부러 비워 둔 칸**이다. step2가 이 파일을 열어 그 자리를 채운다. 두 노트북이 이 파일 하나로 이어진다.

`text, raw = read_image(...)` 줄은 ⑤에서 두 개를 돌려주기로 한 것을 두 이름으로 나눠 받는 것이다.

In [ ]:
import os
import json

os.makedirs(OUTPUT_DIR, exist_ok=True)
done = 0

for i, name in enumerate(images, 1):
    file_id = name.split('.')[0]
    json_path = os.path.join(OUTPUT_DIR, file_id + '.json')
    full_path = os.path.join(OUTPUT_DIR, file_id + '_full.json')

    # 이미 처리한 장은 건너뛴다
    if os.path.exists(json_path):
        print('[%d/%d]' % (i, len(images)), name, '— 이미 처리됨')
        continue

    print('[%d/%d]' % (i, len(images)), name)

    text, raw = read_image(os.path.join(INPUT_DIR, name))

    # (1) 사람이 읽을 파일
    result = {
        'file_path': name,
        'vision_raw': text,
        'gemini_corrected': ''
    }
    with open(json_path, 'w', encoding='utf-8') as f:
        json.dump(result, f, ensure_ascii=False, indent=2)

    # (2) 구글 답장 전체 (좌표·신뢰도)
    with open(full_path, 'w', encoding='utf-8') as f:
        f.write(raw)

    print('   ->', len(text), '자')
    done += 1

print('\n완료:', done, '장 처리')

### ⑧ [실행만] 결과 보기 — **확인 3**

결과가 제대로 만들어졌는지 보고, 첫 장의 텍스트를 화면에 띄운다.

```
결과 파일: 3 개 / 이미지 3 장

E006-005-001-001.json → 412 자
E006-005-001-002.json → 23 자
E006-005-002-001.json → 1038 자

--- E006-005-001-001 미리보기 ---
(뽑힌 텍스트가 여기 나온다)
```

**결과 파일 수와 이미지 수가 같으면 통과다.** `_full.json`은 세지 않는다.

미리보기는 이 실습에서 가장 중요한 화면이다. **원본 이미지를 옆에 띄우고 나란히 본다.** 행 순서가 흐트러진 곳, 모양이 비슷한 글자를 잘못 읽은 곳, 그리고 **원본에 없는 글자가 들어간 곳**을 찾는다. 마지막 것이 환각이며, 원본과 대조하지 않으면 드러나지 않는다.

지금 결과는 **글자를 뽑아낸 것이지 읽어 낸 것이 아니다.** 왜 다음 단계가 필요한지 이 화면에서 보인다.

In [ ]:
import os
import json

# _full.json 은 빼고 센다
result_files = []
for name in sorted(os.listdir(OUTPUT_DIR)):
    if name.endswith('.json') and not name.endswith('_full.json'):
        result_files.append(name)

print('결과 파일:', len(result_files), '개 / 이미지', len(images), '장')
print()

for name in result_files:
    data = json.load(open(os.path.join(OUTPUT_DIR, name), encoding='utf-8'))
    print(name, '→', len(data['vision_raw']), '자')

# 첫 장 미리보기
first = json.load(open(os.path.join(OUTPUT_DIR, result_files[0]), encoding='utf-8'))

print()
print('---', result_files[0].split('.')[0], '미리보기 ---')
print(first['vision_raw'][:500])

### ⑨ *[참고]* 신뢰도 살펴보기 — 시간이 남을 때만

**여기까지 안 해도 step2로 넘어갈 수 있다.** 진도가 빠듯하면 건너뛰고, 강의 뒤에 혼자 열어 봐도 된다.

⑦에서 만든 `_full.json`에 무엇이 들었는지 보는 자리다. 두 가지가 들어 있다.

* **좌표** — 그 글자가 이미지의 어느 자리에 있었는지. 왼쪽에서 320픽셀, 위에서 88픽셀 하는 식이다. 이게 있으면 나중에 판독문의 한 글자를 눌렀을 때 원본 사진의 그 자리가 표시되게 만들 수 있다.
* **신뢰도** — 모델이 자기 판독을 얼마나 확신하는지. 0에서 1 사이 숫자로, 0.98이면 거의 확신, 0.42면 자신 없음이다.

아래를 실행하면 **모델이 가장 자신 없어 한 글자 10개**가 나온다.

```
--- E006-005-001-001 ---
전체 단어: 218 개

모델이 자신 없어 한 것 10개
0.31 蒙
0.38 螭
0.44 舊
...
```

**이 화면이 검토의 출발점이다.** 900장을 처음부터 훑는 것은 불가능하지만, 모델이 스스로 표시해 준 데부터 원본과 대조하는 것은 할 수 있다. 사람이 어디를 봐야 하는지를 기계가 알려 주는 셈이다.

---

코드에 `for`가 **네 겹**으로 겹쳐 있다. 답장이 상자 안에 상자가 든 꼴이기 때문이다.

```
페이지 → 블록 → 문단 → 단어 → 글자
```

한 장(페이지) 안에 여러 덩어리(블록)가 있고, 덩어리 안에 문단이, 문단 안에 단어가, 단어 안에 글자가 있다. 겉의 `for`가 바깥 상자를 하나씩 열고, 안의 `for`가 그 안을 다시 연다. **코드를 다 읽을 필요는 없다.** 이런 모양으로 생겼다는 것만 보면 된다.

In [ ]:
import os
import json

# 첫 장의 _full.json 을 연다
full_files = []
for name in sorted(os.listdir(OUTPUT_DIR)):
    if name.endswith('_full.json'):
        full_files.append(name)

target = full_files[0]
data = json.load(open(os.path.join(OUTPUT_DIR, target), encoding='utf-8'))

print('---', target.replace('_full.json', ''), '---')

annotation = data.get('fullTextAnnotation')

if annotation is None:
    print('이 장에서는 글자가 잡히지 않았습니다.')
else:
    words = []

    # 상자 안의 상자를 하나씩 연다
    for page in annotation['pages']:
        for block in page['blocks']:
            for paragraph in block['paragraphs']:
                for word in paragraph['words']:

                    letters = ''
                    for symbol in word['symbols']:
                        letters = letters + symbol['text']

                    score = word.get('confidence', 0)
                    words.append([score, letters])

    words.sort()   # 신뢰도가 낮은 것부터 앞으로

    print('전체 단어:', len(words), '개')
    print()
    print('모델이 자신 없어 한 것 10개')

    for score, letters in words[:10]:
        print(round(score, 2), letters)

### ⑩ 다음 단계

사료 한 장마다 파일 둘이 결과 폴더에 만들어졌다. 짧은 쪽에는 방금 뽑은 `vision_raw`와 아직 비어 있는 `gemini_corrected`가 들어 있고, 긴 쪽에는 좌표와 신뢰도가 들어 있다.

**이 결과는 아직 판독문이 아니다.** 글자가 뽑혔을 뿐, 행 순서가 흐트러져 있고 잘못 읽거나 없는 글자를 만들어 낸 곳이 남아 있다. 문맥에 맞게 교정하는 작업은 step2에서 이어진다.

→ [Step 2 — Gemini 교정](https://github.com/Song-yiJung/korean-ocr-lectures/blob/main/2026-08-aks-lecture/02_step2_gemini/step2_gemini_colab_20260813.ipynb)

step2에서는 ③의 `INPUT_DIR`·`OUTPUT_DIR` 두 값을 **여기서 쓴 것과 글자 그대로 같게** 넣어야 한다. 이 노트북을 닫지 말고 열어 두면 옮겨 적기 편하다.